In [0]:
# Step 4.1 — Setup for Silver Cleaning

from pyspark.sql.functions import *
from pyspark.sql.types import *

target_schema = "workspace.census360"

# Bronze table names
bronze_gn_population_table = f"{target_schema}.bronze_gn_population"
bronze_gn_housing_table = f"{target_schema}.bronze_gn_housing"
bronze_gnd_list_table = f"{target_schema}.bronze_gnd_list"
bronze_trcsl_table = f"{target_schema}.bronze_trcsl_telecom_statistics"
bronze_district_population_table = f"{target_schema}.bronze_population_by_district_census_years"
bronze_population_2012_table = f"{target_schema}.bronze_population_2012"

print("Silver cleaning notebook setup completed.")

In [0]:
# Step 4.2 — Load Bronze Tables

bronze_gn_population = spark.table(bronze_gn_population_table)
bronze_gn_housing = spark.table(bronze_gn_housing_table)
bronze_gnd_list = spark.table(bronze_gnd_list_table)
bronze_trcsl = spark.table(bronze_trcsl_table)
bronze_district_population = spark.table(bronze_district_population_table)
bronze_population_2012 = spark.table(bronze_population_2012_table)

print("Bronze tables loaded successfully.")

print("GN Population rows:", bronze_gn_population.count())
print("GN Housing rows:", bronze_gn_housing.count())
print("GND List rows:", bronze_gnd_list.count())
print("TRCSL rows:", bronze_trcsl.count())
print("District population history rows:", bronze_district_population.count())
print("Population 2012 rows:", bronze_population_2012.count())

In [0]:
# Step 4.3 — Clean GN Population and Save as Silver Table

silver_gn_population = (
    bronze_gn_population
    .select(
        col("province_code").cast("string").alias("province_code"),
        trim(col("province_name")).alias("province_name"),

        col("district_code").cast("string").alias("district_code"),
        trim(col("district_name")).alias("district_name"),

        col("ds_division_code").cast("string").alias("ds_division_code"),
        trim(col("ds_division_name")).alias("ds_division_name"),

        col("gn_division_code").cast("string").alias("gn_division_code"),
        trim(col("gn_division_name")).alias("gn_division_name"),
        col("gn_division_number").cast("string").alias("gn_division_number"),

        col("population_total").cast("int").alias("population_total"),
        col("male_population").cast("int").alias("male_population"),
        col("female_population").cast("int").alias("female_population"),
        col("age_total").cast("int").alias("age_total"),
        col("age_0_14").cast("int").alias("age_0_14"),
        col("age_15_59").cast("int").alias("age_15_59"),
        col("age_60_64").cast("int").alias("age_60_64"),
        col("age_65_plus").cast("int").alias("age_65_plus")
    )
    .filter(col("gn_division_code").isNotNull())
    .filter(col("gn_division_name").isNotNull())
    .dropDuplicates(["district_code", "ds_division_code", "gn_division_code"])
)

spark.sql(f"DROP TABLE IF EXISTS {target_schema}.silver_gn_population")

silver_gn_population.write.mode("overwrite").saveAsTable(
    f"{target_schema}.silver_gn_population"
)

print("silver_gn_population created successfully.")
print("Rows:", silver_gn_population.count())

display(silver_gn_population.limit(10))

In [0]:
# Step 4.4 — Clean GN Housing and Save as Silver Table

silver_gn_housing = (
    bronze_gn_housing
    .select(
        col("province_code").cast("string").alias("province_code"),
        trim(col("province_name")).alias("province_name"),

        col("district_code").cast("string").alias("district_code"),
        trim(col("district_name")).alias("district_name"),

        col("ds_division_code").cast("string").alias("ds_division_code"),
        trim(col("ds_division_name")).alias("ds_division_name"),

        col("gn_division_code").cast("string").alias("gn_division_code"),
        trim(col("gn_division_name")).alias("gn_division_name"),
        col("gn_division_number").cast("string").alias("gn_division_number"),

        col("occupied_housing_units").cast("int").alias("occupied_housing_units")
    )
    .filter(col("gn_division_code").isNotNull())
    .filter(col("gn_division_name").isNotNull())
    .dropDuplicates(["district_code", "ds_division_code", "gn_division_code"])
)

spark.sql(f"DROP TABLE IF EXISTS {target_schema}.silver_gn_housing")

silver_gn_housing.write.mode("overwrite").saveAsTable(
    f"{target_schema}.silver_gn_housing"
)

print("silver_gn_housing created successfully.")
print("Rows:", silver_gn_housing.count())

display(silver_gn_housing.limit(10))

In [0]:
# Step 4.5 — Clean GND Master and Save as Silver Table

silver_gnd_master = (
    bronze_gnd_list
    .select(
        col("province_code").cast("string").alias("province_code"),
        trim(col("province_name")).alias("province_name"),

        col("district_code").cast("string").alias("district_code"),
        trim(col("district_name")).alias("district_name"),

        col("dsd__code").cast("string").alias("ds_division_code"),
        trim(col("dsd_name")).alias("ds_division_name"),

        col("gnd__code").cast("string").alias("gn_division_code"),
        col("gnd_num").cast("string").alias("gn_division_number"),
        trim(col("gnd_name")).alias("gn_division_name"),

        col("lg_code").cast("string").alias("local_government_code"),
        trim(col("lg_name")).alias("local_government_name")
    )
    .filter(col("gn_division_code").isNotNull())
    .filter(col("gn_division_name").isNotNull())
    .dropDuplicates(["district_code", "ds_division_code", "gn_division_code"])
)

spark.sql(f"DROP TABLE IF EXISTS {target_schema}.silver_gnd_master")

silver_gnd_master.write.mode("overwrite").saveAsTable(
    f"{target_schema}.silver_gnd_master"
)

print("silver_gnd_master created successfully.")
print("Rows:", silver_gnd_master.count())

display(silver_gnd_master.limit(10))

In [0]:
# Step 4.6 — Create GN Demographic Base Table

silver_gn_demographic_base = (
    silver_gnd_master.alias("m")
    .join(
        silver_gn_population.alias("p"),
        on=[
            col("m.district_code") == col("p.district_code"),
            col("m.ds_division_code") == col("p.ds_division_code"),
            col("m.gn_division_code") == col("p.gn_division_code")
        ],
        how="left"
    )
    .join(
        silver_gn_housing.alias("h"),
        on=[
            col("m.district_code") == col("h.district_code"),
            col("m.ds_division_code") == col("h.ds_division_code"),
            col("m.gn_division_code") == col("h.gn_division_code")
        ],
        how="left"
    )
    .select(
        col("m.province_code"),
        col("m.province_name"),
        col("m.district_code"),
        col("m.district_name"),
        col("m.ds_division_code"),
        col("m.ds_division_name"),
        col("m.gn_division_code"),
        col("m.gn_division_number"),
        col("m.gn_division_name"),
        col("m.local_government_code"),
        col("m.local_government_name"),

        col("p.population_total"),
        col("p.male_population"),
        col("p.female_population"),
        col("p.age_total"),
        col("p.age_0_14"),
        col("p.age_15_59"),
        col("p.age_60_64"),
        col("p.age_65_plus"),

        col("h.occupied_housing_units")
    )
    .dropDuplicates(["district_code", "ds_division_code", "gn_division_code"])
)

spark.sql(f"DROP TABLE IF EXISTS {target_schema}.silver_gn_demographic_base")

silver_gn_demographic_base.write.mode("overwrite").saveAsTable(
    f"{target_schema}.silver_gn_demographic_base"
)

print("silver_gn_demographic_base created successfully.")
print("Rows:", silver_gn_demographic_base.count())

display(silver_gn_demographic_base.limit(10))

In [0]:
# Step 4.5A — Diagnose GND Master issue

print("Bronze GND List rows:", bronze_gnd_list.count())

print("Bronze GND List columns:")
print(bronze_gnd_list.columns)

print("Null count checks:")
bronze_gnd_list.select(
    count("*").alias("total_rows"),
    countDistinct("gnd_uid").alias("distinct_gnd_uid"),
    countDistinct("district_code", "dsd__code", "gnd__code").alias("distinct_district_dsd_gnd_code"),
    countDistinct("district_code", "dsd_name", "gnd_name").alias("distinct_district_dsd_gnd_name")
).show(truncate=False)

print("Sample GND rows:")
display(
    bronze_gnd_list.select(
        "serial_number",
        "gnd_uid",
        "province_code",
        "province_name",
        "district_code",
        "district_name",
        "dsd__code",
        "dsd_name",
        "gnd__code",
        "gnd_num",
        "gnd_name",
        "lg_code",
        "lg_name"
    ).limit(20)
)

In [0]:
# Step 4.5B — Rebuild GND Master using GND_UID as unique key

silver_gnd_master = (
    bronze_gnd_list
    .select(
        col("gnd_uid").cast("string").alias("gnd_uid"),

        col("province_code").cast("string").alias("province_code"),
        trim(col("province_name")).alias("province_name"),

        col("district_code").cast("string").alias("district_code"),
        trim(col("district_name")).alias("district_name"),

        col("dsd__code").cast("string").alias("ds_division_code"),
        trim(col("dsd_name")).alias("ds_division_name"),

        col("gnd__code").cast("string").alias("gn_division_code"),
        col("gnd_num").cast("string").alias("gn_division_number"),
        trim(col("gnd_name")).alias("gn_division_name"),

        col("local_government_code") if "local_government_code" in bronze_gnd_list.columns else col("lg_code").cast("string").alias("local_government_code"),
        col("local_government_name") if "local_government_name" in bronze_gnd_list.columns else trim(col("lg_name")).alias("local_government_name")
    )
    .filter(col("gnd_uid").isNotNull())
    .filter(col("gn_division_name").isNotNull())
    .dropDuplicates(["gnd_uid"])
)

spark.sql(f"DROP TABLE IF EXISTS {target_schema}.silver_gnd_master")

silver_gnd_master.write.mode("overwrite").saveAsTable(
    f"{target_schema}.silver_gnd_master"
)

print("silver_gnd_master fixed successfully.")
print("Rows:", silver_gnd_master.count())

display(silver_gnd_master.limit(10))

In [0]:
# Check fixed Silver GND master count

spark.table(f"{target_schema}.silver_gnd_master").count()

In [0]:
# Step 4.6B — Rebuild GN Demographic Base using safer name-based join keys

# Reload fixed Silver tables from catalog
silver_gnd_master = spark.table(f"{target_schema}.silver_gnd_master")
silver_gn_population = spark.table(f"{target_schema}.silver_gn_population")
silver_gn_housing = spark.table(f"{target_schema}.silver_gn_housing")

# Add normalized join keys to reduce mismatch caused by spaces/case differences
m = (
    silver_gnd_master
    .withColumn("join_district_name", lower(trim(col("district_name"))))
    .withColumn("join_ds_name", lower(trim(col("ds_division_name"))))
    .withColumn("join_gn_name", lower(trim(col("gn_division_name"))))
)

p = (
    silver_gn_population
    .withColumn("join_district_name", lower(trim(col("district_name"))))
    .withColumn("join_ds_name", lower(trim(col("ds_division_name"))))
    .withColumn("join_gn_name", lower(trim(col("gn_division_name"))))
)

h = (
    silver_gn_housing
    .withColumn("join_district_name", lower(trim(col("district_name"))))
    .withColumn("join_ds_name", lower(trim(col("ds_division_name"))))
    .withColumn("join_gn_name", lower(trim(col("gn_division_name"))))
)

silver_gn_demographic_base = (
    m.alias("m")
    .join(
        p.alias("p"),
        on=[
            col("m.join_district_name") == col("p.join_district_name"),
            col("m.join_ds_name") == col("p.join_ds_name"),
            col("m.join_gn_name") == col("p.join_gn_name")
        ],
        how="left"
    )
    .join(
        h.alias("h"),
        on=[
            col("m.join_district_name") == col("h.join_district_name"),
            col("m.join_ds_name") == col("h.join_ds_name"),
            col("m.join_gn_name") == col("h.join_gn_name")
        ],
        how="left"
    )
    .select(
        col("m.gnd_uid"),
        col("m.province_code"),
        col("m.province_name"),
        col("m.district_code"),
        col("m.district_name"),
        col("m.ds_division_code"),
        col("m.ds_division_name"),
        col("m.gn_division_code"),
        col("m.gn_division_number"),
        col("m.gn_division_name"),
        col("m.local_government_code"),
        col("m.local_government_name"),

        col("p.population_total"),
        col("p.male_population"),
        col("p.female_population"),
        col("p.age_total"),
        col("p.age_0_14"),
        col("p.age_15_59"),
        col("p.age_60_64"),
        col("p.age_65_plus"),

        col("h.occupied_housing_units")
    )
    .dropDuplicates(["gnd_uid"])
)

spark.sql(f"DROP TABLE IF EXISTS {target_schema}.silver_gn_demographic_base")

silver_gn_demographic_base.write.mode("overwrite").saveAsTable(
    f"{target_schema}.silver_gn_demographic_base"
)

print("silver_gn_demographic_base rebuilt successfully.")
print("Rows:", silver_gn_demographic_base.count())

display(silver_gn_demographic_base.limit(10))

In [0]:
# Step 4.7 — Validate Silver GN Demographic Base

silver_gn_demographic_base = spark.table(f"{target_schema}.silver_gn_demographic_base")

print("Total rows:", silver_gn_demographic_base.count())

silver_gn_demographic_base.select(
    count("*").alias("total_rows"),
    countDistinct("gnd_uid").alias("distinct_gnd_uid"),
    countDistinct("district_name").alias("district_count"),
    countDistinct("ds_division_name").alias("ds_division_count"),
    countDistinct("gn_division_name").alias("gn_name_count"),
    sum(when(col("population_total").isNull(), 1).otherwise(0)).alias("missing_population_rows"),
    sum(when(col("occupied_housing_units").isNull(), 1).otherwise(0)).alias("missing_housing_rows")
).show(truncate=False)

display(
    silver_gn_demographic_base
    .select(
        "gnd_uid",
        "province_name",
        "district_name",
        "ds_division_name",
        "gn_division_name",
        "population_total",
        "occupied_housing_units"
    )
    .limit(20)
)

In [0]:
# Step 4.8 — Clean TRCSL Telecom Statistics and Save as Silver Table

print("Bronze TRCSL columns:")
print(bronze_trcsl.columns)

display(bronze_trcsl)

In [0]:
# Step 4.8B — Clean TRCSL Telecom Statistics and Save as Silver Table

silver_trcsl_telecom_statistics = (
    bronze_trcsl
    .select(
        col("year").cast("int").alias("year"),
        col("quarter").cast("int").alias("quarter"),
        trim(col("period_label")).alias("period_label"),

        col("mobile_subscriptions").cast("long").alias("mobile_subscriptions"),
        col("mobile_density_per_100").cast("double").alias("mobile_density_per_100"),

        col("mobile_broadband_subscriptions").cast("long").alias("mobile_broadband_subscriptions"),
        col("mobile_broadband_density_per_100").cast("double").alias("mobile_broadband_density_per_100"),

        col("fixed_access_subscriptions").cast("long").alias("fixed_access_subscriptions"),
        col("fixed_broadband_subscriptions").cast("long").alias("fixed_broadband_subscriptions"),
        col("fixed_broadband_density_per_100").cast("double").alias("fixed_broadband_density_per_100"),

        col("satellite_broadband_subscriptions").cast("long").alias("satellite_broadband_subscriptions"),
        trim(col("notes")).alias("notes")
    )
    .filter(col("year").isNotNull())
    .dropDuplicates(["year", "quarter"])
)

spark.sql(f"DROP TABLE IF EXISTS {target_schema}.silver_trcsl_telecom_statistics")

silver_trcsl_telecom_statistics.write.mode("overwrite").saveAsTable(
    f"{target_schema}.silver_trcsl_telecom_statistics"
)

print("silver_trcsl_telecom_statistics created successfully.")
print("Rows:", silver_trcsl_telecom_statistics.count())

display(silver_trcsl_telecom_statistics)

In [0]:
# Step 4.8C — Clean TRCSL Telecom Statistics and Save as Silver Table
# Fix: use try_cast because the Bronze table contains an extra header row like 'Year'

silver_trcsl_telecom_statistics = (
    bronze_trcsl
    .select(
        expr("try_cast(year as int)").alias("year"),
        expr("try_cast(quarter as int)").alias("quarter"),
        trim(col("period_label")).alias("period_label"),

        expr("try_cast(mobile_subscriptions as bigint)").alias("mobile_subscriptions"),
        expr("try_cast(mobile_density_per_100 as double)").alias("mobile_density_per_100"),

        expr("try_cast(mobile_broadband_subscriptions as bigint)").alias("mobile_broadband_subscriptions"),
        expr("try_cast(mobile_broadband_density_per_100 as double)").alias("mobile_broadband_density_per_100"),

        expr("try_cast(fixed_access_subscriptions as bigint)").alias("fixed_access_subscriptions"),
        expr("try_cast(fixed_broadband_subscriptions as bigint)").alias("fixed_broadband_subscriptions"),
        expr("try_cast(fixed_broadband_density_per_100 as double)").alias("fixed_broadband_density_per_100"),

        expr("try_cast(satellite_broadband_subscriptions as bigint)").alias("satellite_broadband_subscriptions"),
        trim(col("notes")).alias("notes")
    )
    .filter(col("year").isNotNull())
    .dropDuplicates(["year", "quarter"])
)

spark.sql(f"DROP TABLE IF EXISTS {target_schema}.silver_trcsl_telecom_statistics")

silver_trcsl_telecom_statistics.write.mode("overwrite").saveAsTable(
    f"{target_schema}.silver_trcsl_telecom_statistics"
)

print("silver_trcsl_telecom_statistics created successfully.")
print("Rows:", silver_trcsl_telecom_statistics.count())

display(silver_trcsl_telecom_statistics)

In [0]:
# Step 4.9 — Inspect District Population History Bronze Table

print("Bronze district population columns:")
print(bronze_district_population.columns)

print("Rows:", bronze_district_population.count())

display(bronze_district_population.limit(10))

In [0]:
# Step 4.9B — Clean District Population History and Save as Silver Table

year_columns = [
    "1871", "1881", "1891", "1901", "1911", "1921", "1931",
    "1946", "1953", "1963", "1971", "1981", "2001", "2012"
]

# Convert wide year columns into long format
stack_expr = "stack({0}, {1}) as (census_year, population)".format(
    len(year_columns),
    ", ".join([f"'{y}', `{y}`" for y in year_columns])
)

silver_district_population_history = (
    bronze_district_population
    .select(
        trim(col("district")).alias("district_name"),
        expr(stack_expr)
    )
    .select(
        trim(col("district_name")).alias("district_name"),
        col("census_year").cast("int").alias("census_year"),
        expr("try_cast(population as bigint)").alias("population")
    )
    .filter(col("district_name").isNotNull())
    .filter(col("census_year").isNotNull())
    .filter(col("population").isNotNull())
)

spark.sql(f"DROP TABLE IF EXISTS {target_schema}.silver_district_population_history")

silver_district_population_history.write.mode("overwrite").saveAsTable(
    f"{target_schema}.silver_district_population_history"
)

print("silver_district_population_history created successfully.")
print("Rows:", silver_district_population_history.count())

display(
    silver_district_population_history
    .orderBy("district_name", "census_year")
    .limit(30)
)

In [0]:
# Step 4.10 — Inspect Population 2012 Bronze Table

print("Bronze population 2012 columns:")
print(bronze_population_2012.columns)

print("Rows:", bronze_population_2012.count())

display(bronze_population_2012.limit(10))

In [0]:
# Step 4.10B — Clean Population 2012 and Save as Silver Table

silver_population_2012 = (
    bronze_population_2012
    .select(
        trim(col("district")).alias("district_name"),

        expr("try_cast(total_number_of_persons as bigint)").alias("population_total_2012"),
        expr("try_cast(male as bigint)").alias("male_population_2012"),
        expr("try_cast(female as bigint)").alias("female_population_2012"),

        expr("try_cast(less_than_15_years as bigint)").alias("age_less_than_15_2012"),
        expr("try_cast(15__59_years as bigint)").alias("age_15_59_2012"),
        expr("try_cast(60_years_and_over as bigint)").alias("age_60_plus_2012")
    )
    .filter(col("district_name").isNotNull())
    .filter(col("population_total_2012").isNotNull())
    .dropDuplicates(["district_name"])
)

spark.sql(f"DROP TABLE IF EXISTS {target_schema}.silver_population_2012")

silver_population_2012.write.mode("overwrite").saveAsTable(
    f"{target_schema}.silver_population_2012"
)

print("silver_population_2012 created successfully.")
print("Rows:", silver_population_2012.count())

display(silver_population_2012.orderBy("district_name"))

In [0]:
# Step 4.11 — Final Silver Layer Validation

silver_tables = [
    "silver_gn_population",
    "silver_gn_housing",
    "silver_gnd_master",
    "silver_gn_demographic_base",
    "silver_trcsl_telecom_statistics",
    "silver_district_population_history",
    "silver_population_2012"
]

for table in silver_tables:
    full_table_name = f"{target_schema}.{table}"
    row_count = spark.table(full_table_name).count()
    print(f"{full_table_name}: {row_count} rows")